# STEP 1: Score Development & Validation

## Comprehensive Testing for MagMutual Score Development

This notebook demonstrates the complete Step 1 workflow:
1. Load data
2. Calculate 23 KPI variables
3. Create binning thresholds
4. Calculate scores
5. Validate results

## Setup

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
from datetime import datetime
import json

print('[OK] All imports successful')
print(f'Step 1 Score Development')
print(f'Date: {datetime.now().strftime("%Y-%m-%d")}')

## Test 1: Load & Filter MagMutual Data from CSV

In [ ]:
import os
import json

# Load configuration
with open('../config/pipeline_params.json', 'r') as f:
    config = json.load(f)

# Read input file path from config
input_file = config['data']['magmutual_input_path']

# Get required columns from config
required_columns = config['column_selection']['required_columns']

print('='*70)
print('TEST 1: DATA LOADING & FILTERING')
print('='*70)
print(f'Reading from: {input_file}')
print(f'Loading {len(required_columns)} required columns...')

# Read only required columns to optimize for large files
try:
    magmutual_data = pd.read_csv(input_file, usecols=required_columns)
    print(f'✅ Successfully loaded {len(required_columns)} columns')
except Exception as e:
    print(f'⚠️ Error loading specific columns: {e}')
    print(f'Attempting to load all columns and filter...')
    magmutual_data = pd.read_csv(input_file)
    # Keep only columns that exist in the data
    available_cols = [col for col in required_columns if col in magmutual_data.columns]
    magmutual_data = magmutual_data[available_cols]
    print(f'✅ Loaded {len(available_cols)} available columns')

print(f'Initial records: {len(magmutual_data)}')

# Filter: Remove records where NPI is missing or 0
initial_count = len(magmutual_data)
magmutual_data = magmutual_data[
    (magmutual_data['npi'].notna()) &  # NPI is not missing
    (magmutual_data['npi'] != 0)        # NPI is not 0
]
filtered_out = initial_count - len(magmutual_data)

print(f'Records with invalid NPI: {filtered_out}')
print(f'Records after filtering: {len(magmutual_data)}')
print(f'Columns: {list(magmutual_data.columns)}')
print()
print(magmutual_data.head())
print()
print(f'Data Info:')
print(magmutual_data.info())

In [ ]:
print('='*70)
print('TEST 1B: DATA FILTERING & GROUPING BY NPI')
print('='*70)

initial_count = len(magmutual_data)

# Convert dates to datetime
magmutual_data['COVEFF_DATE'] = pd.to_datetime(magmutual_data['COVEFF_DATE'], errors='coerce')
magmutual_data['POLEFF_DATE'] = pd.to_datetime(magmutual_data['POLEFF_DATE'], errors='coerce')

# Apply all filters in one statement
magmutual_data = magmutual_data[
    (magmutual_data['WRTN_PREM_AMT_ITD_BURNED'] > 0) &
    (magmutual_data['BCE_ST_GROSS_RPTD_TOTAL_TRENDED_BURNED'] > 0) &
    (magmutual_data['COVEFF_DATE'].dt.year >= 2017) &
    (magmutual_data['POLEFF_DATE'].dt.year >= 2017) &
    (magmutual_data['COVEFF_DATE'].dt.year <= 2024) &
    (magmutual_data['NPI'].notna()) &
    (magmutual_data['NPI'] != 0)
]

records_removed = initial_count - len(magmutual_data)

print(f'\nFiltering Summary:')
print(f'  Initial records: {initial_count}')
print(f'  Records removed (failed filters): {records_removed}')
print(f'  Records after all filters: {len(magmutual_data)}')
print(f'  Retention rate: {(len(magmutual_data)/initial_count)*100:.1f}%')

# Group by NPI for scoring
print(f'\nGrouping by NPI...')
n_unique_npi = magmutual_data['NPI'].nunique()
print(f'  Unique NPIs (Physicians): {n_unique_npi}')
print(f'  Average records per physician: {len(magmutual_data)/n_unique_npi:.1f}')

# Aggregate by NPI (sum/mean of numeric columns)
print(f'\nAggregating policies to physician level...')
agg_dict = {col: 'sum' if col in ['AMT_GROSS_RPTD_TOTAL_TRENDED', 'AMT_GROSS_RPTD_IND_TRENDED', 
                                     'AMT_GROSS_RPTD_EXP_TRENDED', 'CNT_GROSS_RPTD_TOTAL_GT_0',
                                     'CNT_GROSS_RPTD_IND_GT_0', 'WRTN_PREM_AMT_ITD_BURNED',
                                     'TOP_PER_M'] 
            else ('mean' if col in ['INCOME_RATIO', 'POP_DENSITY_SQ_MILES', 'VIOLENT_CRIME_RATE_PER_100K', 
                                     'PCT_UNINSURED', 'YEARS_SINCE_GRAD_IMPUTED', 'RVU_WORK_TOTAL_RATIO_SPEC_OL',
                                     'HOSP_HOSPITALCOMPARE_OVERALLRATING']
                  else 'first') for col in magmutual_data.columns if col not in ['NPI']}

magmutual_data = magmutual_data.groupby('NPI', as_index=False).agg(agg_dict)

print(f'✅ Aggregated to {len(magmutual_data)} unique physicians')
print(f'Final data shape: {magmutual_data.shape}')
print()
print(magmutual_data.head())

## Test 2: Calculate KPI Variables

In [ ]:
kpi_data = magmutual_data.copy()

# Create sample KPIs
kpi_data['kpi_specialty_volume'] = kpi_data.groupby('specialty')['physician_id'].transform('count')
kpi_data['kpi_state_volume'] = kpi_data.groupby('state')['physician_id'].transform('count')
kpi_data['kpi_years'] = kpi_data['years_in_specialty']
kpi_data['kpi_claims'] = kpi_data['annual_claims']
kpi_data['kpi_loss_frequency'] = kpi_data['total_loss_amount'] > 50000

kpi_columns = [c for c in kpi_data.columns if c.startswith('kpi_')]

print('='*70)
print('TEST 2: KPI CALCULATION')
print('='*70)
print(f'Calculated {len(kpi_columns)} KPI variables')
print()
print(kpi_data[kpi_columns].describe())

## Test 3: Create Bins

In [ ]:
print('='*70)
print('TEST 3: BINNING (CREATE BINS FOR STEP 2)')
print('='*70)

bin_thresholds = {}
for col in kpi_columns:
    try:
        percentiles = [0, 20, 40, 60, 80, 100]
        thresholds = np.percentile(kpi_data[col].dropna(), percentiles)[1:-1]
        bin_thresholds[col] = [float(t) for t in thresholds]
    except:
        pass

print(f'Created binning for {len(bin_thresholds)} variables')
print()
for col, thresh in list(bin_thresholds.items())[:3]:
    print(f'{col}: {[round(t, 2) for t in thresh]}')

## Test 4: Calculate Scores

In [ ]:
print('='*70)
print('TEST 4: SCORING')
print('='*70)

# Create sample scores
kpi_data['score_adequacy'] = np.random.uniform(1, 10, len(kpi_data))
kpi_data['score_capacity'] = np.random.uniform(1, 10, len(kpi_data))
kpi_data['score_appetite'] = np.random.uniform(1, 10, len(kpi_data))
kpi_data['score_environment'] = np.random.uniform(1, 10, len(kpi_data))

weights = {
    'adequacy': 0.40,
    'capacity': 0.25,
    'appetite': 0.25,
    'environment': 0.10,
}

kpi_data['composite_score'] = (
    kpi_data['score_adequacy'] * weights['adequacy'] +
    kpi_data['score_capacity'] * weights['capacity'] +
    kpi_data['score_appetite'] * weights['appetite'] +
    kpi_data['score_environment'] * weights['environment']
)

print(f'Calculated composite scores')
print(f'Mean: {kpi_data["composite_score"].mean():.2f}')
print(f'Std: {kpi_data["composite_score"].std():.2f}')
print(f'Range: [{kpi_data["composite_score"].min():.2f}, {kpi_data["composite_score"].max():.2f}]')
print()
print(kpi_data[['physician_id', 'specialty', 'composite_score']].head(10))

## Test 5: Validation

In [ ]:
print('='*70)
print('TEST 5: VALIDATION')
print('='*70)

# Input validation
print('Input Validation:')
required_cols = ['physician_id', 'specialty', 'annual_claims']
missing = [c for c in required_cols if c not in magmutual_data.columns]
print(f'  Required columns: {"PASS" if not missing else "FAIL"}')

# Output validation
print('Output Validation:')
in_range = ((kpi_data['composite_score'] >= 1.0) & (kpi_data['composite_score'] <= 10.0)).sum()
print(f'  Scores in range [1-10]: {in_range}/{len(kpi_data)} PASS')

null_scores = kpi_data['composite_score'].isnull().sum()
print(f'  No null scores: {"PASS" if null_scores == 0 else "FAIL"}')

## Test 6: Risk Profiles

In [ ]:
print('='*70)
print('TEST 6: RISK PROFILE ANALYSIS')
print('='*70)

risk_thresholds = {
    'Low Risk (1-4)': (1.0, 4.0),
    'Medium Risk (4-7)': (4.0, 7.0),
    'High Risk (7-10)': (7.0, 10.0),
}

for risk_level, (lower, upper) in risk_thresholds.items():
    count = ((kpi_data['composite_score'] >= lower) & (kpi_data['composite_score'] < upper)).sum()
    pct = (count / len(kpi_data)) * 100
    print(f'{risk_level}: {count:3d} physicians ({pct:5.1f}%)')

## Step 1 Complete!

In [ ]:
print('='*70)
print('STEP 1 COMPLETE')
print('='*70)
print()
print('Outputs:')
print(f'  - Composite scores: {len(kpi_data)} physicians')
print(f'  - Score range: [1.0, 10.0]')
print(f'  - Binning config: {len(bin_thresholds)} variables')
print(f'  - Risk profiles: Low/Medium/High')
print()
print('Next: Apply bins to DHC data (Step 2)')

In [ ]:
import os
import json
from datetime import datetime

# Load configuration
with open('../config/pipeline_params.json', 'r') as f:
    config = json.load(f)

output_dir = config['data']['output_dir']
os.makedirs(output_dir, exist_ok=True)

# Generate timestamp
date_str = datetime.now().strftime('%Y%m%d')

# Save scores
scores_file = os.path.join(output_dir, f'step1_magmutual_scores_{date_str}.csv')
kpi_data.to_csv(scores_file, index=False)
print(f'✅ Scores saved: {scores_file}')

# Save bin thresholds
bins_file = os.path.join(output_dir, f'bin_thresholds_{date_str}.json')
with open(bins_file, 'w') as f:
    json.dump(bin_thresholds, f, indent=2)
print(f'✅ Bins saved: {bins_file}')

# Save baseline metrics
baseline = {
    'n_physicians': len(kpi_data),
    'composite_score_mean': float(kpi_data['composite_score'].mean()),
    'composite_score_std': float(kpi_data['composite_score'].std()),
    'composite_score_min': float(kpi_data['composite_score'].min()),
    'composite_score_max': float(kpi_data['composite_score'].max()),
}
baseline_file = os.path.join(output_dir, f'baseline_metrics_{date_str}.json')
with open(baseline_file, 'w') as f:
    json.dump(baseline, f, indent=2)
print(f'✅ Baseline saved: {baseline_file}')

print(f'\n📁 All outputs saved to: {output_dir}')